In [1]:
import pandas as pd
final = pd.read_parquet("../data/processed/modeling_table.parquet")
final_clean = final.dropna(subset=["is_delayed"]).copy()

In [2]:
majority_baseline = final_clean["is_delayed"].value_counts(normalize=True).max()
print(f"Majority-class baseline accuracy: {majority_baseline:.3f}")

Majority-class baseline accuracy: 0.777


In [3]:
route_rates = final_clean.groupby("route_short_name")["is_delayed"].transform("mean")
route_baseline_preds = (route_rates > 0.5).astype(int)
route_baseline_acc = (route_baseline_preds == final_clean["is_delayed"]).mean()
print(f"Route-based naive baseline accuracy: {route_baseline_acc:.3f}")

Route-based naive baseline accuracy: 0.778


In [4]:
print((route_rates > 0.5).mean())  # what fraction of ALL rows get flagged "predicted delayed"
print(final_clean.groupby("route_short_name")["is_delayed"].mean().describe())

0.004658160381608731
count    561.000000
mean       0.252816
std        0.141470
min        0.000000
25%        0.161565
50%        0.235561
75%        0.317179
max        1.000000
Name: is_delayed, dtype: float64


Baselines: majority-class baseline is 77.7% accuracy (all-not-delayed), this differs from the ~85% implied by the August-only sample in Phase 2, since adding winter dates raised the overall delay rate to ~22.3%. A naive route-based baseline (predict delayed if a route's historical rate > 50%) barely improves on this (77.8%), not because route doesn't matter (Phase 3 showed an 11-23x rate gap), but because almost no individual route actually crosses the 50% threshold (mean per-route rate is 25.3%, 75th percentile only 31.7%). This shows route needs to be used as a continuous/weighted signal, not a hard cutoff.